# Second Kaggle run: diversity, context window, perplexity, attacks, human texts

Settings: GPU T4 x2, internet on, secret `GITHUB_TOKEN`, and the output of the first run added as input (Add Input > Your Work > the first notebook).

In [ ]:
import glob
import os
import shutil
import subprocess
import sys
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!git clone -q https://{token}@github.com/omarht27/textgrain-eval.git
%cd /kaggle/working/textgrain-eval
!git remote set-url origin https://github.com/omarht27/textgrain-eval.git
!pip install -q -e . --no-deps
sys.path.insert(0, "/kaggle/working/textgrain-eval/src")

previous = os.path.dirname(glob.glob("/kaggle/input/**/results/secret.key", recursive=True)[0])
shutil.copytree(previous, "results", dirs_exist_ok=True)
!ls results

## Smoke tests (a few minutes): each new script on a handful of texts

In [ ]:
!python experiments/generate.py --model Qwen/Qwen2.5-1.5B-Instruct --prompts questions --n-prompts 2 --samples 2 --max-new-tokens 30 --method textgrain --out results/smoke_questions.jsonl
!python experiments/perplexity.py results/smoke.jsonl
!python experiments/attack.py results/smoke.jsonl --n-texts 2
import json
print(json.load(open("results/smoke.attacks.json"))[0]["texts"])

In [ ]:
MODEL = "--model Qwen/Qwen2.5-1.5B-Instruct --batch-size 16"
QUESTIONS = f"{MODEL} --prompts questions --n-prompts 30 --samples 10 --max-new-tokens 150"
C4 = f"{MODEL} --n-prompts 200 --max-new-tokens 400 --no-eos"
OLD = " ".join(sorted(glob.glob("results/*.jsonl")))
ATTACKED = "results/gumbel.jsonl results/greenlist_d2.jsonl results/textgrain_B4_b0.25.jsonl results/textgrain_B8_b0.25.jsonl"

JOBS = {
    0: [
        f"python experiments/generate.py {QUESTIONS} --method none --out results/questions_none.jsonl",
        f"python experiments/generate.py {QUESTIONS} --method gumbel --out results/questions_gumbel.jsonl",
        f"python experiments/generate.py {QUESTIONS} --method greenlist --out results/questions_greenlist_d2.jsonl",
        f"python experiments/generate.py {QUESTIONS} --method textgrain --beta 0.25 --out results/questions_textgrain_B4_b0.25.jsonl",
        f"python experiments/generate.py {QUESTIONS} --method textgrain --beta 0.25 --blocks 8 --columns 8 --out results/questions_textgrain_B8_b0.25.jsonl",
        f"python experiments/generate.py {QUESTIONS} --method textgrain --beta 0.5 --blocks 8 --columns 8 --out results/questions_textgrain_B8_b0.5.jsonl",
        f"python experiments/generate.py {C4} --method textgrain --beta 0.25 --blocks 8 --columns 8 --window 1 --out results/textgrain_B8_b0.25_L1.jsonl",
        f"python experiments/generate.py {C4} --method textgrain --beta 0.25 --blocks 8 --columns 8 --window 4 --out results/textgrain_B8_b0.25_L4.jsonl",
    ],
    1: [
        f"python experiments/perplexity.py {OLD}",
        f"python experiments/attack.py {ATTACKED}",
        "python experiments/human_texts.py",
    ],
}

processes = []
for gpu, jobs in JOBS.items():
    log = open(f"results/log2_gpu{gpu}.txt", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    processes.append(subprocess.Popen("; ".join(jobs), shell=True, env=env, stdout=log, stderr=subprocess.STDOUT))

Progress (re-run to refresh):

In [ ]:
!tail -n 3 results/log2_gpu0.txt results/log2_gpu1.txt

Wait for both queues, then pack the results:

In [ ]:
for p in processes:
    p.wait()
!tail -n 20 results/log2_gpu0.txt results/log2_gpu1.txt
!zip -q -r results.zip results
!ls -la results.zip